In [1]:
import os
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gdown
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, EsmForMaskedLM

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.decomposition import PCA

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [5]:
#TODO: Add constants from code below

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

In [ ]:
from huggingface_hub import login
# Add your HF token here for faster sequence generation.
login(token="hf_token")

In [3]:
# The data that is used by the model.
train_df = pd.read_csv('https://raw.githubusercontent.com/Shengqing/CHE1148_Team_Protein_Origami/refs/heads/main/data/processed/tsuboyama_processed_train_sampled.csv')
val_df   = pd.read_csv('https://raw.githubusercontent.com/Shengqing/CHE1148_Team_Protein_Origami/refs/heads/main/data/processed/tsuboyama_processed_val_full.csv')

In [6]:
from transformers import AutoTokenizer, EsmForMaskedLM
import pandas as pd

AA = "ACDEFGHIKLMNPQRSTVWY"

def build_aa_token_map(tokenizer):
    return {aa: tokenizer.convert_tokens_to_ids(aa) for aa in AA}

def load_model(device="cuda" if torch.cuda.is_available() else "cpu"):
    model = EsmForMaskedLM.from_pretrained("facebook/esm2_t6_8M_UR50D")
    tokenizer = AutoTokenizer.from_pretrained("facebook/esm2_t6_8M_UR50D")
    print(f"Running on: {device}")
    return model.to(device).eval(), tokenizer, device

def apply_mutation(seq, mut):
    orig_aa, new_aa = mut[0], mut[-1]
    pos = int(mut[1:-1]) - 1
    assert seq[pos] == orig_aa, f"Expected {orig_aa} at pos {pos}, got {seq[pos]}"
    return seq[:pos] + new_aa + seq[pos+1:]

def best_mutation(seq, model, tokenizer, device, aa_token_map):
    ids = tokenizer(seq, return_tensors="pt").to(device)
    input_ids = ids["input_ids"]
    seq_len = len(seq)

    batch = input_ids.repeat(seq_len, 1)
    for i in range(seq_len):
        batch[i, i + 1] = tokenizer.mask_token_id

    with torch.no_grad():
        logits = model(input_ids=batch).logits

    best_score, best_mut = float('-inf'), None

    for i in range(seq_len):
        log_probs = torch.log_softmax(logits[i, i + 1], dim=0)
        for aa in AA:
            if aa == seq[i]:
                continue
            score = log_probs[aa_token_map[aa]].item()
            if score > best_score:
                best_score, best_mut = score, f"{seq[i]}{i+1}{aa}"

    return best_mut, best_score

def process_df(df, out_path, model, tokenizer, device, aa_token_map, n_samples=None):
    if n_samples is not None:
        df = df.head(n_samples)
        print(f"Testing mode: using {n_samples} samples only.")

    records = []
    mutation_summary = []

    for i, row in enumerate(df.itertuples(index=False)):
        seq = str(row.aa_seq).strip()
        print(f"[{i+1}/{len(df)}] {seq[:20]}...")

        mut, score = best_mutation(seq, model, tokenizer, device, aa_token_map)
        mutated_seq = apply_mutation(seq, mut)

        records.append({
            "aa_seq":            seq,
            "mutated_seq":       mutated_seq,
            "deltaG":            float(row.deltaG),
            "deltaG_95CI_high":  float(row.deltaG_95CI_high),
            "deltaG_95CI_low":   float(row.deltaG_95CI_low),
            "WT_cluster":        str(row.WT_cluster),
            "mut_type":          str(row.mut_type),
            "Stabilizing_mut":   bool(row.Stabilizing_mut),
            "best_mutation":     mut,
            "best_score":        score,
        })

        mutation_summary.append({
            "mutated_seq":       mutated_seq,
            "position":          int(mut[1:-1]),
            "letter_change":     f"{mut[0]}->{mut[-1]}",
            "mut_type":          str(row.mut_type),       # <-- added
            "WT_cluster":        str(row.WT_cluster),     # <-- added
            "Stabilizing_mut":   bool(row.Stabilizing_mut),  # <-- added
        })

    summary_df = pd.DataFrame(mutation_summary).set_index("mutated_seq")

    torch.save(records, out_path)
    print(f"Saved {len(records)} records to {out_path}")
    return records, summary_df

# --- Run ---
model, tokenizer, device = load_model()
aa_token_map = build_aa_token_map(tokenizer)

N_SAMPLES = None  # <-- set to None for full dataset

train_records, train_summary_df = process_df(train_df, "train_best_mutations.pt", model, tokenizer, device, aa_token_map, n_samples=N_SAMPLES)

Loading weights:   0%|          | 0/112 [00:00<?, ?it/s]

EsmForMaskedLM LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                         | Status     |  | 
----------------------------+------------+--+-
esm.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Streaming output truncated to the last 5000 lines.
[55002/60000] TTYKLIANGKTCKGETTTEA...
[55003/60000] GLWRFLLSKGRALYNWAKSH...
[55004/60000] SQSVVATQLIPMNTALTPAM...
[55005/60000] QIFVKTLTGKTITLEVEPSD...
[55006/60000] AEKTGIVNVSSSLNAREGAG...
[55007/60000] AVSDRLIGRKGVVMEWISPQ...
[55008/60000] KELVSALYDYQEKSPREVTM...
[55009/60000] MQIFVKTLTGKTITLEVEPS...
[55010/60000] KVTIVVENIKVFGEDGKLTD...
[55011/60000] PEDLERKVRELQKNGVSPEQ...
[55012/60000] MTYKLILNGKTLKGETTTEA...
[55013/60000] SPEVQIAILTEQINNLNEHL...
[55014/60000] FNLKQAKEEAIKELVDAGIA...
[55015/60000] DVEEQIRRLEEVLKKNQPVT...
[55016/60000] GTLHLNGVTVKVPSLEKANK...
[55017/60000] MIINNLKLIREKKKISQSEL...
[55018/60000] DDVELRDGDWRVELKDADEV...
[55019/60000] ITIHLHDGTLHIELTGIDEM...
[55020/60000] NNDALSPAIRRLLAEHNLDA...
[55021/60000] MKVIFLKDVKGKGKKGEIKN...
[55022/60000] STVKVRLGHLEVTLHNVSEE...
[55023/60000] YNLQKLLAPYHKAKTLERQV...
[55024/60000] HSHMSHTQVIELERKGSHQK...
[55025/60000] MGTYKLISNGKTLKGETTTE...
[55026/60000] VTYHVNRYTYHFDNPEEAEK...

In [7]:
train_summary_df.to_csv("train_summary.csv")

In [8]:
train_summary_df.head(
)

,position,letter_change,mut_type,WT_cluster,Stabilizing_mut
mutated_seq,,,,,
MQESVVAAVLHPINTALTVGMMTTRVVSPTGAPAEDIPRLISMQVNQVVPMGTTLMPDMVKGYAP,1,N->M,I11H,23,True
MNAAQIVDEALNQGITLFVADNRLQYETSRDNIPEELDNEWKYYRQDLIDFLQQLD,1,K->M,L38D,30,True
MITITWHGKEQAMKAMKKMQAQNLEVHVRVENGQWVITAD,1,R->M,T6W,EHEE,True
MTLFVASYDYEVRTEDDLSFHKGEKFQILNSSEGDWWEARSLTTGETGYIPSNYVAPV,1,V->M,A12V,15,True
MKKMAKAIMADPNKADEVYKKWATKGYTLTQMSNFLKSKTAGKYDRVYNGYVIHLD,1,V->M,D24T,16,True
